In [6]:
import json
import base64
from openai import OpenAI

In [7]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [8]:
QUESTION_BANK = [
    ("What colour is the {tool}?",
     ["silver/grey", "white", "black", "gold", "other"]),
    ("Is the {tool} on the left or right side of the image?",
     ["left", "right", "centre"]),
    ("Which direction is the tip of the {tool} pointing?",
     ["up", "down", "left", "right"]),
    ("How many surgical instruments are visible in the image?",
     ["1", "2", "3", "4+"]),
]

In [9]:
SYSTEM_PROMPT = """\
You are shown ONE laparoscopic cholecystectomy frame and told which surgical
tool is present in it. Answer SIMPLE, visually-grounded questions about that tool
- the kind anyone could answer by looking, with no medical knowledge.
 
For EACH question you are given an exact list of allowed answers. You MUST choose
one answer from that list and nothing else. If the relevant part of the tool is
not visible enough to judge, choose "not_visible" when it is offered; otherwise
choose the closest allowed answer. 
 
Answer only from what is plainly visible in this single frame. Do not use surgical
knowledge, do not guess about anatomy, do not describe actions over time.
 
OUTPUT - return ONLY valid JSON, no preamble, no markdown fences:
{"answers": [{"question": "...", "answer": "..."}, ...]}
"""
 

In [10]:
def _parse_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [11]:
def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [ ]:
 
def answer_grounded(image_path: str, tool: str) -> list[dict]:
    """Answer the fixed question bank about `tool` for one frame.
    """
    # Build the question list with allowed answers spelled out for this tool.
    lines = []
    for q_tmpl, allowed in QUESTION_BANK:
        q = q_tmpl.format(tool=tool)
        lines.append(f'- "{q}"  allowed answers: {allowed}')
    bank_text = "\n".join(lines)
 
    user_text = (
        f"Tool present in this frame: {tool}\n\n"
        f"Answer each of these questions, one entry per question:\n{bank_text}"
    )
 
    b64 = _encode_image(image_path)

    
 
    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=SYSTEM_PROMPT,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": user_text},
                {"type": "input_image",
                 "image_url": f"data:image/png;base64,{b64}"},
            ],
        }],
    )
 
    try:
        answers = _parse_json(response.output_text)["answers"]
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Parse failed for {image_path} ({tool}): {e}")
        return []
 
    # Keep only confident, in-set answers.
    allowed_by_q = {q_tmpl.format(tool=tool): set(a) for q_tmpl, a in QUESTION_BANK}
    kept = []
    for item in answers:
        q, a = item.get("question"), item.get("answer")
        if q in allowed_by_q and a in allowed_by_q[q]:
            kept.append(item)
    return kept

In [13]:
import json
import re
with open('/home/as5606/Datasets/cholec_formatted_data/cholec80_llava_test.json', 'r') as f:
    data = json.load(f)

In [22]:
from tqdm import tqdm

data_2 = []

print("Processing samples...")
for item in tqdm(data, desc="Processing"):
    answer = item['conversations'][1]['value'].lower()

    # Only get 'is used' (not 'is not used')
    if 'is used' in answer and 'is not used' not in answer:
        # Extract tool (word before "is")
        match = re.search(r'(\w+)\s+is', answer)
        if match:
            tool = match.group(1)
            
            try:
                # Get grounded questions
                result = answer_grounded(item['image'], tool=tool)
                
                if result:
                    # Create new entry with original data + grounded questions
                    new_entry = {**item}
                    new_entry['grounded_qa_pairs'] = result
                    data_2.append(new_entry)
                    
            except Exception as e:
                print(f"Error processing {item['id']}: {e}")
        else:
            print(f"Tool not found in answer: {answer}")


Processing samples...


Processing:   2%|▏         | 143/7652 [00:10<08:54, 14.04it/s]


KeyboardInterrupt: 

In [23]:
data_2

[{'id': 'video02_frame001200',
  'image': '/home/as5606/Datasets/chole80_framed/cholec80/frames/video02/video02_000049.png',
  'conversations': [{'from': 'human',
    'value': 'is scissors used in preparation?\n<image>'},
   {'from': 'gpt', 'value': 'scissors is used in preparation'}],
  'grounded_qa_pairs': [{'question': 'What colour is the scissors?',
    'answer': 'silver/grey'},
   {'question': 'Is the scissors on the left or right side of the image?',
    'answer': 'right'},
   {'question': 'Which direction is the tip of the scissors pointing?',
    'answer': 'left'},
   {'question': 'How many surgical instruments are visible in the image?',
    'answer': '1'}]},
 {'id': 'video02_frame004800',
  'image': '/home/as5606/Datasets/chole80_framed/cholec80/frames/video02/video02_000193.png',
  'conversations': [{'from': 'human',
    'value': 'is scissors used in preparation?\n<image>'},
   {'from': 'gpt', 'value': 'scissors is used in preparation'}],
  'grounded_qa_pairs': [{'question':